In [368]:
import torch
import torch.nn as nn
import numpy as np

In [369]:
def im2col_multi(X, kernel_shape, stride=1, padding=(0, 0)):
    B, C = X.shape[:2]
    #print(f'B: {B}, C: {C}')
    kH, kW = kernel_shape

    if isinstance(padding, tuple):
        pad_H, pad_W = padding
    else:
        pad_H = pad_W = padding

    X_padded = np.pad(X, ( (0, 0), (0, 0),(pad_H, pad_H), (pad_W, pad_W) ), mode='constant')

    H_p, W_p = X_padded.shape[2:]

    out_H = (H_p- kH) // stride + 1
    out_W = (W_p- kW) // stride + 1

    L = out_H * out_W

    # ----- Allocate output -----
    # PyTorch shape: (B, C*kH*kW, L)
    cols = np.zeros((B, C * kH * kW, L))

    patch_idx = 0
    for i in range(out_H):
        for j in range(out_W):

            h_start = i * stride
            w_start = j * stride

            # slice patch for ALL channels
            patch = X_padded[:, :, h_start:h_start + kH, w_start:w_start + kW]

            # flatten channels + kernel dims
            patch = patch.reshape(B, -1)    # → (B, C*kH*kW)

            cols[:, :, patch_idx] = patch
            patch_idx += 1
    
    return cols, out_H, out_W

In [ ]:
torch.manual_seed(1)

unfold = nn.Unfold(kernel_size=(2, 2), stride=1, padding=0)
input = torch.tensor([[[[1, 2, 3, 4, 5],
                        [6, 7, 8, 9, 10],
                        [11, 12, 13, 14, 15],
                        [16, 17, 18, 19, 20],
                        [21, 22, 23, 24, 25]],]], dtype=torch.float32)


#input = torch.randn(30, 3, 28, 28)
print(input.shape)
output = unfold(input)
print(output.shape)

torch.Size([1, 1, 5, 5])
torch.Size([1, 4, 16])


In [371]:
new, outh, outw = im2col_multi(input, kernel_shape=(2, 2), stride=1, padding=0)

print(new.shape)

(1, 4, 16)


In [372]:
print(output)

tensor([[[ 1.,  2.,  3.,  4.,  6.,  7.,  8.,  9., 11., 12., 13., 14., 16., 17.,
          18., 19.],
         [ 2.,  3.,  4.,  5.,  7.,  8.,  9., 10., 12., 13., 14., 15., 17., 18.,
          19., 20.],
         [ 6.,  7.,  8.,  9., 11., 12., 13., 14., 16., 17., 18., 19., 21., 22.,
          23., 24.],
         [ 7.,  8.,  9., 10., 12., 13., 14., 15., 17., 18., 19., 20., 22., 23.,
          24., 25.]]])


In [375]:
print(new.T)

[[[ 1.]
  [ 2.]
  [ 6.]
  [ 7.]]

 [[ 2.]
  [ 3.]
  [ 7.]
  [ 8.]]

 [[ 3.]
  [ 4.]
  [ 8.]
  [ 9.]]

 [[ 4.]
  [ 5.]
  [ 9.]
  [10.]]

 [[ 6.]
  [ 7.]
  [11.]
  [12.]]

 [[ 7.]
  [ 8.]
  [12.]
  [13.]]

 [[ 8.]
  [ 9.]
  [13.]
  [14.]]

 [[ 9.]
  [10.]
  [14.]
  [15.]]

 [[11.]
  [12.]
  [16.]
  [17.]]

 [[12.]
  [13.]
  [17.]
  [18.]]

 [[13.]
  [14.]
  [18.]
  [19.]]

 [[14.]
  [15.]
  [19.]
  [20.]]

 [[16.]
  [17.]
  [21.]
  [22.]]

 [[17.]
  [18.]
  [22.]
  [23.]]

 [[18.]
  [19.]
  [23.]
  [24.]]

 [[19.]
  [20.]
  [24.]
  [25.]]]


In [374]:
np.allclose(output, new)

True